In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
llm_api_key = os.getenv("LLM_API_KEY")
llm_base_url = os.getenv("LLM_BASE_URL")

In [ ]:
from openai import OpenAI


def get_llm_section_response(
    model_name: str,
    system_prompt: str,
    section_prompt: str,
) -> str:
    """
    Send a single ECHR section prompt to the LLM and return its response.
    """

    api_key = os.getenv("GWDG_API_KEY")
    if not api_key:
        raise RuntimeError("GWDG_API_KEY not found in environment variables")


    client = OpenAI(api_key=llm_api_key, base_url=llm_base_url)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": section_prompt},
    ]

    chat_completion = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.1,
        top_p=1,
    )

    return chat_completion.choices[0].message.content.strip()


In [11]:
import json
with open("intermediate_data/section_prompts.json", "r", encoding="utf-8") as f:
    section_prompts = json.load(f)


In [13]:
from typing import Dict, Any
import traceback


def get_llm_responses_for_all_sections(
    model_name: str,
    system_prompt: str,
    section_prompts: Dict[str, str],
    max_retries: int = 1,
) -> Dict[str, Any]:
    """
    Get LLM responses for all ECHR sections in a fault-tolerant way.

    - Continues if one section fails
    - Captures error details per section
    - Optional retries per section

    Returns:
    {
        "responses": {
            "A. Applicant": "<LLM response>",
            ...
        },
        "errors": {
            "B. States": {
                "error": "Exception message",
                "traceback": "..."
            },
            ...
        }
    }
    """

    responses: Dict[str, str] = {}
    errors: Dict[str, Dict[str, str]] = {}

    for section_name, section_prompt in section_prompts.items():
        attempt = 0
        success = False

        while attempt <= max_retries and not success:
            try:
                response = get_llm_section_response(
                    model_name=model_name,
                    system_prompt=system_prompt,
                    section_prompt=section_prompt,
                )
                responses[section_name] = response
                success = True

            except Exception as exc:
                attempt += 1

                # If retries exhausted, record error
                if attempt > max_retries:
                    errors[section_name] = {
                        "error": str(exc),
                        "traceback": traceback.format_exc(),
                    }

    return {
        "responses": responses,
        "errors": errors,
    }

In [14]:
system_prompt = (
    "You are a legal assistant helping to complete an "
    "European Court of Human Rights application form.\n"
    "You must return strictly valid JSON only."
)

result = get_llm_responses_for_all_sections(
    model_name="gpt-4o-mini",
    system_prompt=system_prompt,
    section_prompts=section_prompts,
    max_retries=2,
)

successful = result["responses"]
failed = result["errors"]

print(successful.keys())
print(failed.keys())

dict_keys([])
dict_keys(['A. Applicant', 'B. States', 'C. Representative (Individual applicant)', 'D. Representative (Organisation)', 'E. Statement of facts', 'F. Alleged violations', 'G. Admissibility', 'H. Other international proceedings', 'I. List of accompanying documents', 'J. Final declaration'])
